<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Bridge_Evals_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Running HuggingFace-style Evals on TransformerBridge

This notebook closes [#107](https://github.com/TransformerLensOrg/TransformerLens/issues/107) by showing how to run benchmark-style evaluations on a `TransformerBridge` model. There are two paths, and which one you want depends on whether you need TL hooks active during the eval:

- **Path A — pass the underlying HF model to an existing eval framework.** `bridge.original_model` is a stock `transformers.AutoModelForCausalLM`, so `lm-eval-harness` and HF `evaluate` work out of the box without any TL-specific glue.
- **Path B — manual loop with hooks active.** This is the interpretability use case: run a benchmark *while an intervention is applied* (ablation, patching, steering vector) and measure the accuracy delta. Bridge supports `hooks()` / `run_with_hooks()` natively, so the eval loop is a few lines on top of `bridge.forward()`.

## Setup

In [1]:
# NBVAL_IGNORE_OUTPUT
import os

IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except ImportError:
    IN_COLAB = False

if not IN_GITHUB and not IN_COLAB:
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install transformer_lens
    %pip install lm-eval

Running as a Jupyter notebook - intended for development only!


In [ ]:
# NBVAL_IGNORE_OUTPUT
import torch

from transformer_lens.model_bridge import TransformerBridge

## Load the model

We use `gpt2` so the notebook stays light and runs in CI without a GPU.

In [3]:
# NBVAL_IGNORE_OUTPUT
bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")
bridge.eval();

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Path A — plug into an existing HF eval framework

`bridge.original_model` is a real `transformers.AutoModelForCausalLM` and `bridge.tokenizer` is the matching HF tokenizer. Any framework that accepts an HF model + tokenizer accepts the bridge:

In [4]:
from transformers import PreTrainedModel, PreTrainedTokenizerBase

assert isinstance(bridge.original_model, PreTrainedModel)
assert isinstance(bridge.tokenizer, PreTrainedTokenizerBase)
print(type(bridge.original_model).__name__, "/", type(bridge.tokenizer).__name__)

GPT2LMHeadModel / GPT2Tokenizer


### Smoke test against `lm-eval-harness`

Wrap the bridge's HF model in `HFLM` and call `loglikelihood` directly — this is the primitive every multiple-choice eval (PIQA, HellaSwag, ARC, …) builds on. We use a synthetic `(context, continuation)` pair so no dataset download is needed.

When you run the next cell, `lm-eval-harness` will print two stderr warnings about passing a non-string `pretrained` arg ("Many other model arguments may be ignored" / "assuming single-process call"). These are expected and benign — they fire whenever an existing model object is wired in instead of letting `HFLM` load by name, which is exactly what we want here.

In [5]:
# NBVAL_IGNORE_OUTPUT
from lm_eval.api.instance import Instance
from lm_eval.models.huggingface import HFLM

lm = HFLM(pretrained=bridge.original_model, tokenizer=bridge.tokenizer)
request = Instance(
    request_type="loglikelihood",
    doc={},
    arguments=("Once upon a", " time"),
    idx=0,
)
(logprob, is_greedy), = lm.loglikelihood([request])
assert is_greedy, "gpt2 should greedily continue 'Once upon a' with ' time'"
assert logprob > -0.5, f"unexpectedly low logprob: {logprob}"
print(f"is_greedy={is_greedy}")

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.


Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration



Running loglikelihood requests:   0%|          | 0/1 [00:00<?, ?it/s]


Running loglikelihood requests: 100%|██████████| 1/1 [00:00<00:00, 25.44it/s]

is_greedy=True


From here a full benchmark is one call (commented out — downloads the dataset):

```python
from lm_eval import simple_evaluate
results = simple_evaluate(model=lm, tasks=["lambada_openai", "piqa"])
```

HF `evaluate` works the same way — anywhere a `PreTrainedModel` is accepted, pass `bridge.original_model`.

## Path B — manual eval loop with hooks active

When the eval needs to run *while an intervention is applied* (ablate a head, patch an activation, add a steering vector), the manual loop is the clearer pattern. Below is a minimal LAMBADA-style next-token accuracy harness using `bridge.forward()` and `bridge.hooks()`.

In [6]:
# An inline mini eval set: (prefix, expected next token). These are completions
# gpt2-small actually nails, so we have a clean baseline to perturb. Replace
# with `datasets.load_dataset("EleutherAI/lambada_openai", "en")` for the real
# benchmark.
examples = [
    ("Once upon a", " time"),
    ("To be, or not to", " be"),
    ("A B C D E F", " G"),
    ("I love you, and you love", " me"),
    ("one, two, three, four,", " five"),
    ("Happy birthday to", " you"),
]


def next_token_accuracy(model: TransformerBridge, eval_set) -> float:
    correct = 0
    for prefix, target in eval_set:
        tokens = model.to_tokens(prefix)
        with torch.no_grad():
            logits = model(tokens)
        pred_id = logits[0, -1].argmax().item()
        target_id = model.to_single_token(target)
        correct += int(pred_id == target_id)
    return correct / len(eval_set)

In [7]:
baseline = next_token_accuracy(bridge, examples)
print(f"baseline accuracy: {baseline:.2f}")

baseline accuracy: 1.00


### Re-run the same eval with an intervention active

Here we zero out the last block's MLP output for the duration of the eval. The hook stays active for every example inside the `with` block, so `next_token_accuracy` doesn't need to know anything about the intervention.

In [8]:
def zero_last_mlp(out, hook):
    out[...] = 0
    return out


with bridge.hooks(fwd_hooks=[("blocks.11.hook_mlp_out", zero_last_mlp)]):
    ablated = next_token_accuracy(bridge, examples)

print(f"ablated accuracy:  {ablated:.2f}")
print(f"delta:             {ablated - baseline:+.2f}")

ablated accuracy:  0.83
delta:             -0.17


## Recap

- For standard benchmarks, hand `bridge.original_model` + `bridge.tokenizer` to whatever framework you already use (`lm-eval-harness`, HF `evaluate`, etc.).
- For evaluations *with hooks*, write a small accuracy function over `bridge.forward()` and wrap calls in `bridge.hooks(...)` (or call `bridge.run_with_hooks(...)`).

If you hit a gap with a specific eval framework, please open a focused issue.